<a href="https://colab.research.google.com/github/McKendreevv/Mis433/blob/main/Competition_MVV.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [34]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

In [35]:
from google.colab import drive

drive.mount('/content/drive')

output_dir = '/content/drive/My Drive/Colab Notebooks/output/'
os.makedirs(output_dir, exist_ok=True)
print(f'Submission files will be saved to: {output_dir}')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Submission files will be saved to: /content/drive/My Drive/Colab Notebooks/output/


In [36]:
accounts = pd.read_csv('/content/drive/My Drive/Colab Notebooks/data/accounts.csv')
testset = pd.read_csv('/content/drive/My Drive/Colab Notebooks/data/churn_test.csv')
trainset = pd.read_csv('/content/drive/My Drive/Colab Notebooks/data/churn_train.csv')
demographics = pd.read_csv('/content/drive/My Drive/Colab Notebooks/data/demographics.csv')
services = pd.read_csv('/content/drive/My Drive/Colab Notebooks/data/services.csv')

In [37]:
accounts.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 5298 entries, 0 to 5297
Data columns (total 8 columns):
 #   Column             Non-Null Count  Dtype  
---  ------             --------------  -----  
 0   CustomerNo         5298 non-null   int64  
 1   BaseCharges        5298 non-null   float64
 2   DOC                5298 non-null   object 
 3   TotalCharges       5288 non-null   float64
 4   DOE                5298 non-null   object 
 5   ElectronicBilling  5298 non-null   object 
 6   ContractType       5293 non-null   object 
 7   PaymentMethod      5298 non-null   object 
dtypes: float64(2), int64(1), object(5)
memory usage: 331.3+ KB


In [38]:
testset.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1060 entries, 0 to 1059
Data columns (total 1 columns):
 #   Column      Non-Null Count  Dtype
---  ------      --------------  -----
 0   CustomerNo  1060 non-null   int64
dtypes: int64(1)
memory usage: 8.4 KB


In [39]:
trainset.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 4238 entries, 0 to 4237
Data columns (total 2 columns):
 #   Column      Non-Null Count  Dtype
---  ------      --------------  -----
 0   CustomerNo  4238 non-null   int64
 1   Churn       4238 non-null   int64
dtypes: int64(2)
memory usage: 66.3 KB


In [40]:
demographics.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 5298 entries, 0 to 5297
Data columns (total 8 columns):
 #   Column         Non-Null Count  Dtype 
---  ------         --------------  ----- 
 0   CustomerNo     5298 non-null   int64 
 1   Country        5298 non-null   object
 2   State          5298 non-null   object
 3   Retired        5298 non-null   int64 
 4   HasPartner     5298 non-null   int64 
 5   HasDependents  5298 non-null   int64 
 6   Education      5288 non-null   object
 7   Gender         5294 non-null   object
dtypes: int64(4), object(4)
memory usage: 331.3+ KB


In [41]:
services.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 47682 entries, 0 to 47681
Data columns (total 3 columns):
 #   Column         Non-Null Count  Dtype 
---  ------         --------------  ----- 
 0   CustomerNo     47682 non-null  int64 
 1   TypeOfService  47682 non-null  object
 2   SeviceDetails  47682 non-null  object
dtypes: int64(1), object(2)
memory usage: 1.1+ MB


Preprocessing:

In [42]:
# Merge datasets
train = trainset.merge(accounts, on='CustomerNo', how='left')
train = train.merge(demographics, on='CustomerNo', how='left')
train = train.merge(services, on='CustomerNo', how='left')

test = testset.merge(accounts, on='CustomerNo', how='left')
test = test.merge(demographics, on='CustomerNo', how='left')
test = test.merge(services, on='CustomerNo', how='left')

# Keep only one row per customer
train = train.drop_duplicates(subset='CustomerNo')
test = test.drop_duplicates(subset='CustomerNo')

print(train['CustomerNo'].nunique())
print(len(train))

print(test['CustomerNo'].nunique())
print(len(test))


# Target variable
y = train['Churn']

# Remove unnecessary columns
X = train.drop(columns=['CustomerNo', 'Churn'])
X_test = test.drop(columns=['CustomerNo'])

# Combine
combined = pd.concat([X, X_test], axis=0)

# One-hot encoding
combined = pd.get_dummies(combined, drop_first=True)

# Fill missing values
combined = combined.fillna(0)

# Split back
X = combined.iloc[:len(X), :]
X_test = combined.iloc[len(X):, :]

4238
4238
1060
1060


In [43]:
#train
from sklearn.model_selection import train_test_split

X_train, X_valid, y_train, y_valid = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42
)

In [44]:
#standardize
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()

X_train_scaled = scaler.fit_transform(X_train)
X_valid_scaled = scaler.transform(X_valid)
X_test_scaled = scaler.transform(X_test)

In [45]:
#Logistic regression
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score

log_model = LogisticRegression(max_iter=2000)

log_model.fit(X_train_scaled, y_train)

log_preds = log_model.predict(X_valid_scaled)

log_acc = accuracy_score(y_valid, log_preds)

print("Logistic Regression Accuracy:", log_acc)

submission = pd.DataFrame({
    'CustomerNo': test['CustomerNo'],
    'Churn': log_model.predict(X_test_scaled)
})

submission.to_csv(output_dir + 'submission_log2.csv', index=False)


Logistic Regression Accuracy: 0.8089622641509434


In [46]:
#decision tree
from sklearn.tree import DecisionTreeClassifier

tree_model = DecisionTreeClassifier(
    max_depth=5,
    random_state=42
)

tree_model.fit(X_train, y_train)

tree_preds = tree_model.predict(X_valid)

tree_acc = accuracy_score(y_valid, tree_preds)

print("Decision Tree Accuracy:", tree_acc)

submission = pd.DataFrame({
    'CustomerNo': test['CustomerNo'],
    'Churn': tree_model.predict(X_test)
})

submission.to_csv(output_dir + 'submission_tree2.csv', index=False)

Decision Tree Accuracy: 0.7841981132075472


In [47]:
from sklearn.ensemble import RandomForestClassifier

rf_model = RandomForestClassifier(
    n_estimators=300,
    max_depth=10,
    random_state=42
)

rf_model.fit(X_train, y_train)

rf_preds = rf_model.predict(X_valid)

rf_acc = accuracy_score(y_valid, rf_preds)

print("Random Forest Accuracy:", rf_acc)

submission = pd.DataFrame({
    'CustomerNo': test['CustomerNo'],
    'Churn': rf_model.predict(X_test)
})

submission.to_csv(output_dir + 'submission_rf2.csv', index=False)

Random Forest Accuracy: 0.8101415094339622
